# Lesson 8.3 · Forests in practice

The random forest is a model that almost always works "as it is". It has features worth knowing, though: its probabilities are not calibrated, with imbalanced classes you need to pay attention to the threshold, outside the range of the data it is flat like a tree, and it costs memory and prediction time.

**What you will learn**

- How the probabilities of a forest behave and how to correct them.
- Regression with forests and what they cannot do (extrapolate).
- The costs: model size and latency.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.calibration import CalibratedClassifierCV
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import train_test_split
import time, pickle

## Probabilities of a forest

`predict_proba` is the average of the leaf fractions of the trees. With deep trees and almost pure leaves the probabilities are extreme in a single tree, but the **average** softens them: a forest gives reasonable probabilities, but they tend to be **underconfident** at the margins (rarely 0 or 1) and not calibrated.
Post-hoc calibration (`CalibratedClassifierCV`) corrects them.

In [ ]:
X, y = make_classification(6000, n_features=12, n_informative=5, n_redundant=2, class_sep=0.7, flip_y=0.08, random_state=1)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.5, random_state=0)
models = {
    "logistic": LogisticRegression(max_iter=1000),
    "forest": RandomForestClassifier(300, min_samples_leaf=3, random_state=0, n_jobs=-1),
    "calibrated forest": CalibratedClassifierCV(RandomForestClassifier(300, min_samples_leaf=3, random_state=0, n_jobs=-1), method="isotonic", cv=3),
}
for name, m in models.items():
    p = m.fit(X_tr, y_tr).predict_proba(X_te)[:, 1]
    print(f"{name:18s} AUC {roc_auc_score(y_te, p):.3f} | Brier {brier_score_loss(y_te, p):.4f} | log-loss {log_loss(y_te, p):.3f}")

## Regression and limits

`RandomForestRegressor` predicts the average of the leaf means: robust, no scaling, it handles interactions. But it is a **piecewise constant** function and outside the range of the data **it is flat** (like the single tree, M7 C1).

## Cost

A forest with hundreds of deep trees takes up memory (here 11 MB for 100 trees on 15,000 rows, and it grows with the rows: hundreds of MB on large datasets) and prediction requires going through all the trees. To serve the model in real time the number of trees, the depth and the minimum leaf size are cost levers, as well as accuracy levers: with a minimum leaf of 20 the model weighs 3 MB and loses a single point (0.955 → 0.942).

In [ ]:
Xr, yr = make_classification(20000, n_features=20, n_informative=8, random_state=0)
for name, kw in {"100 trees, leaf 1": dict(n_estimators=100), "100 trees, leaf 20": dict(n_estimators=100, min_samples_leaf=20), "30 trees, leaf 20": dict(n_estimators=30, min_samples_leaf=20)}.items():
    f = RandomForestClassifier(random_state=0, n_jobs=-1, **kw).fit(Xr[:15000], yr[:15000])
    t0 = time.time(); f.predict(Xr[15000:]); tp = time.time() - t0
    print(f"{name:24s} size {len(pickle.dumps(f)) / 1e6:7.1f} MB | prediction on 5000 examples {tp:.2f} s | test {f.score(Xr[15000:], yr[15000:]):.3f}")

> **Common mistake.** Treating the probabilities of a forest as calibrated risks (to compute an expected cost, for example) without checking them: use a calibration curve or `CalibratedClassifierCV` (module 4).

## Check your understanding

> *Click on a question to show its answer.*

<details><summary>When do you prefer a forest to a boosting?</summary>

When you want a robust model, not very sensitive to hyperparameters, with free OOB estimates and easy parallel training. Boosting (module 9) usually reaches higher accuracy but requires more care in choosing the learning rate, the depth and the number of trees.

</details>

<details><summary>Can a forest overfit?</summary>

Not by adding trees. It can memorize the training set with deep trees (training accuracy 1.0) without the test getting much worse: that is why you always look at the OOB or the test, never the training. With very noisy data or few examples, a larger minimum leaf helps.

</details>